In [3]:
!pip install pyspark great_expectations pandas

In [4]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet

--2026-04-26 00:43:49--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 54.230.102.132, 54.230.102.184, 54.230.102.151, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|54.230.102.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 47673370 (45M) [application/x-www-form-urlencoded]
Saving to: ‘yellow_tripdata_2023-01.parquet.1’

yellow_tripdata_202 100%[===================>]  45.46M  99.1MB/s    in 0.5s    

2026-04-26 00:43:50 (99.1 MB/s) - ‘yellow_tripdata_2023-01.parquet.1’ saved [47673370/47673370]



In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NYC Taxi Pipeline") \
    .getOrCreate()

In [6]:
raw_df = spark.read.parquet("yellow_tripdata_2023-01.parquet")

raw_df.printSchema()
raw_df.show(5)

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+----

In [7]:
raw_df.write.mode("overwrite").parquet("/content/lake/bronze/taxi_raw")

In [9]:
from pyspark.sql.functions import col

clean_df = raw_df \
    .dropDuplicates() \
    .filter(col("fare_amount") > 0) \
    .filter(col("trip_distance") > 0)

In [10]:
clean_df.write.mode("overwrite").parquet("/content/lake/silver/taxi_clean")

In [11]:
from pyspark.sql.functions import to_date, sum as _sum, count

gold_df = clean_df.withColumn("date", to_date("tpep_pickup_datetime")) \
    .groupBy("date") \
    .agg(
        count("*").alias("total_trips"),
        _sum("fare_amount").alias("total_revenue")
    )

In [12]:
gold_df.write.mode("overwrite").partitionBy("date") \
    .parquet("/content/lake/gold/taxi_analytics")

In [1]:
!pip install great_expectations==0.17.23

In [14]:
import great_expectations as ge

pdf = clean_df.limit(10000).toPandas()

ge_df = ge.from_pandas(pdf)

  return datetime.utcnow().replace(tzinfo=utc)



In [15]:
import great_expectations as ge

context = ge.get_context()

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpbrt4idc0' for ephemeral docs site


In [16]:
pdf = clean_df.limit(10000).toPandas()

  return datetime.utcnow().replace(tzinfo=utc)



In [17]:
from great_expectations.dataset.pandas_dataset import PandasDataset

ge_df = PandasDataset(pdf)

In [18]:
ge_df.expect_column_values_to_not_be_null("fare_amount")

ge_df.expect_column_values_to_be_between(
    "trip_distance", min_value=0, max_value=100
)

ge_df.expect_column_values_to_be_between(
    "fare_amount", min_value=0, max_value=500
)

ge_df.expect_column_values_to_not_be_null("tpep_pickup_datetime")

{
  "success": true,
  "expectation_config": {
    "expectation_type": "expect_column_values_to_not_be_null",
    "kwargs": {
      "column": "tpep_pickup_datetime",
      "result_format": "BASIC"
    },
    "meta": {}
  },
  "result": {
    "element_count": 10000,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "partial_unexpected_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [19]:
results = ge_df.validate()
results

{
  "success": true,
  "results": [
    {
      "success": true,
      "expectation_config": {
        "expectation_type": "expect_column_values_to_not_be_null",
        "kwargs": {
          "column": "fare_amount",
          "result_format": "BASIC"
        },
        "meta": {}
      },
      "result": {
        "element_count": 10000,
        "unexpected_count": 0,
        "unexpected_percent": 0.0,
        "unexpected_percent_total": 0.0,
        "partial_unexpected_list": []
      },
      "meta": {},
      "exception_info": {
        "raised_exception": false,
        "exception_message": null,
        "exception_traceback": null
      }
    },
    {
      "success": true,
      "expectation_config": {
        "expectation_type": "expect_column_values_to_be_between",
        "kwargs": {
          "column": "fare_amount",
          "min_value": 0,
          "max_value": 500,
          "result_format": "BASIC"
        },
        "meta": {}
      },
      "result": {
        "eleme

In [20]:
def ingest():
    print(" Ingesting raw data...")
    return spark.read.parquet("yellow_tripdata_2023-01.parquet")

def silver_layer(df):
    print(" Cleaning data...")
    from pyspark.sql.functions import col
    return df.dropDuplicates() \
             .filter(col("fare_amount") > 0) \
             .filter(col("trip_distance") > 0)

def gold_layer(df):
    print(" Creating analytics layer...")
    from pyspark.sql.functions import to_date, count, sum as _sum

    return df.withColumn("date", to_date("tpep_pickup_datetime")) \
             .groupBy("date") \
             .agg(
                 count("*").alias("trips"),
                 _sum("fare_amount").alias("revenue")
             )

def pipeline():
    raw = ingest()
    clean = silver_layer(raw)
    gold = gold_layer(clean)

    gold.show(5)

pipeline()

 Ingesting raw data...
 Cleaning data...
 Creating analytics layer...
+----------+------+------------------+
|      date| trips|           revenue|
+----------+------+------------------+
|2023-01-01| 74474|1644568.8899999943|
|2023-01-02| 63928|1424169.1999999988|
|2023-01-28|109272| 1882516.939999995|
|2023-01-11|103879|1825658.4100000039|
|2023-02-01|    10|             220.2|
+----------+------+------------------+
only showing top 5 rows


In [21]:
print("Bronze (raw):", raw_df.count())
print("Silver (clean):", clean_df.count())
print("Gold (aggregated):", gold_df.count())

Bronze (raw): 3066766
Silver (clean): 2998382
Gold (aggregated): 35


In [24]:
!zip -r lake_output.zip /content/lake

  adding: content/lake/ (stored 0%)
  adding: content/lake/bronze/ (stored 0%)
  adding: content/lake/bronze/taxi_raw/ (stored 0%)
  adding: content/lake/bronze/taxi_raw/._SUCCESS.crc (stored 0%)
  adding: content/lake/bronze/taxi_raw/.part-00000-ac5815d7-d9a8-4d65-af0e-84560d7c4ba0-c000.snappy.parquet.crc (deflated 0%)
  adding: content/lake/bronze/taxi_raw/_SUCCESS (stored 0%)
  adding: content/lake/bronze/taxi_raw/part-00000-ac5815d7-d9a8-4d65-af0e-84560d7c4ba0-c000.snappy.parquet (deflated 17%)
  adding: content/lake/silver/ (stored 0%)
  adding: content/lake/silver/taxi_clean/ (stored 0%)
  adding: content/lake/silver/taxi_clean/part-00002-595d04f1-e591-4980-b435-2c994605f64c-c000.snappy.parquet (deflated 18%)
  adding: content/lake/silver/taxi_clean/part-00001-595d04f1-e591-4980-b435-2c994605f64c-c000.snappy.parquet (deflated 25%)
  adding: content/lake/silver/taxi_clean/._SUCCESS.crc (stored 0%)
  adding: content/lake/silver/taxi_clean/.part-00001-595d04f1-e591-4980-b435-2c99460

In [25]:
from google.colab import files
files.download("lake_output.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>